# Full Customers Dimension Implementation
This notebook implements the full ingestion and transformation pipeline across **Bronze**, **Silver**, and **Gold** layers for customer dimension data using PySpark and Delta Lake on Databricks.

## 1. Environment Setup & Configuration

In [0]:
%run ../1_setup/utilities

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

# Widgets / Dynamic Parameters
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data source", "customers")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data source")

# -------------------------------------------------------------------------
# CENTRALIZED CONFIGURATION (Centralized parameters for easy maintenance)
# -------------------------------------------------------------------------
STORAGE_ACCOUNT = "fmcgaccount.dfs.core.windows.net"
CONTAINER_NAME = "sports-bar-dp"
PARENT_COMPANY_DIM_TABLE = "fmcg.gold.dim_customers"

# Static Metadata Default Values
DEFAULT_MARKET = "India"
DEFAULT_PLATFORM = "Sports bar"
DEFAULT_CHANNEL = "Aquisition"

# Derived Data Paths and Table Names
base_path = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}/{data_source}/*.csv"
bronze_table_name = f"{catalog}.bronze.{data_source}"
silver_table_name = f"{catalog}.silver.{data_source}"
gold_table_name = f"{catalog}.gold.sb_dim_{data_source}"

print(f"Catalog: {catalog}")
print(f"Data Source: {data_source}")
print(f"Base Path: {base_path}")

## 2. Bronze Layer: Raw Ingestion

In [0]:
# Load raw CSV data with lineage metadata
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

# Write raw ingestion to Bronze Delta table
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(bronze_table_name)
)

## 3. Silver Layer: Cleaning & Normalization

In [0]:
bronze_df = spark.read.table(bronze_table_name)

# Mapping dictionary for city standardization
city_typo_map = {
    # HYDERABAD
    "Hyderbad": "Hyderabad", "Hyderabadd": "Hyderabad", "Hyderabad": "Hyderabad",
    "hyderbad": "Hyderabad", "hyderabadd": "Hyderabad", "hyderabad": "Hyderabad",
    "hydrabad": "Hyderabad", "hyderabd": "Hyderabad", "hyderbd": "Hyderabad",
    "hyderrabad": "Hyderabad", "hyderabadh": "Hyderabad", "hyderbaad": "Hyderabad",
    "hidrabad": "Hyderabad",
    
    # BENGALURU
    "Bengaluruu": "Bengaluru", "Bengalore": "Bengaluru", "Bengaluru": "Bengaluru",
    "bengaluruu": "Bengaluru", "bengalore": "Bengaluru", "bengaluru": "Bengaluru",
    "bangalore": "Bengaluru", "bangaluru": "Bengaluru", "bengalluru": "Bengaluru",
    "bengaloree": "Bengaluru", "bengluru": "Bengaluru", "bengalru": "Bengaluru",
    "bengalaru": "Bengaluru",
    
    # NEW DELHI
    "NewDelhee": "New Delhi", "NewDheli": "New Delhi", "NewDelhi": "New Delhi",
    "New Delhi": "New Delhi", "newdelhee": "New Delhi", "newdheli": "New Delhi",
    "newdelhi": "New Delhi", "new delhi": "New Delhi", "new dheli": "New Delhi",
    "new delhee": "New Delhi", "new deli": "New Delhi", "newdeli": "New Delhi",
    "nu delhi": "New Delhi", "newdehlhi": "New Delhi",
    
    # UNKNOWN / NULL SENTINELS
    "unknown": None, "null": None, "Unknown": None, "UNKNOWN": None,
    "Null": None, "NULL": None, "none": None, "None": None,
    "nan": None, "NaN": None, "": None,
}

silver_df = (
    bronze_df
    # 1. Clean whitespace
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    # 2. Map typos to standard names or None
    .replace(to_replace=city_typo_map, subset=["city"])
    # 3. Handle remaining null/empty values
    .withColumn(
        "city",
        F.when(
            F.col("city").isNull() | (F.col("city") == ""), F.lit("Unknown")
        ).otherwise(F.col("city")),
    )
    # 4. Deduplicate rows
    .dropDuplicates()
    # 5. Format customer attributes
    .withColumn("customer_name", F.initcap(F.col("customer_name")))
    .withColumn("customer_id", F.col("customer_id").cast("string"))
    .withColumn(
        "customer",
        F.concat_ws("-", F.col("customer_name"), F.coalesce(F.col("city"), F.lit("Unknown"))),
    )
    # 6. Apply enterprise default metadata
    .withColumn("market", F.lit(DEFAULT_MARKET))
    .withColumn("platform", F.lit(DEFAULT_PLATFORM))
    .withColumn("channel", F.lit(DEFAULT_CHANNEL))
)

# Write cleaned data to Silver Delta table
(
    silver_df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(silver_table_name)
)

## 4. Gold Layer: Local Dimension Creation

In [0]:
gold_columns = ["customer_id", "customer_name", "city", "customer", "market", "platform", "channel"]
gold_df = silver_df.select(*gold_columns)

# Write to domain-specific Gold table
(
    gold_df.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(gold_table_name)
)

## 5. Enterprise Integration: Upsert into Parent Dimension

In [0]:
customer_column_mapping = {
    "customer_id": "customer_code",
    "customer": "customer",
    "market": "market",
    "platform": "platform",
    "channel": "channel"
}

# Omitting update_set and insert_values defaults to UpdateAll/InsertAll
merge_child_to_parent_dim(
    spark=spark,
    child_table=gold_df,
    parent_table_name=PARENT_COMPANY_DIM_TABLE,
    column_mapping=customer_column_mapping,
    merge_key="customer_code"
)